# Public Mail — automatic full-list collection in Google Colab

**Run all → approve Drive → upload your CSV. No crawl options to change.** A CPU runtime is sufficient. Your PC does not do the scraping.

This version processes the **entire saved website queue**, then automatically runs a deeper contact pass and a bounded transient-error recovery pass. **There is no default 100-site pilot and no five-hour planned stop.** Public Gmail, Yahoo, Outlook, ISP and business-domain addresses are all eligible, with exact source URLs.

### First run
1. Select a **CPU Python runtime** and choose **Runtime → Run all**.
2. Approve the ordinary Google Drive authorization and upload your original **`domains.csv`** when asked.
3. Leave the long-running collection cell working. Progress shows **websites started / never started**, not just email counts. The final ZIP is saved to private Drive and offered for download.

### Continue an existing run — including the old 120-email result
Use the **same Google account and project name `publicmail-33144`**, then **Run all again**. Existing evidence is preserved, the queue is restored, and unfinished websites/passes resume automatically. Do **not** delete your old project, create an empty replacement, or upload a results-only email CSV. If your old run used another project name, use that same name below.

**INCOMPLETE means unfinished** even if some emails were found. A finished automatic pass has no unstarted or queued websites and all three stages recorded as finished. Blocked, failed, no-email and finite-limit outcomes remain explicit in `coverage.csv` and `run_report.json`.

33,144 websites are **not** 33,144 guaranteed email addresses. Unpublished addresses cannot be collected. This notebook never guesses addresses, bypasses robots/CAPTCHA/authentication, or claims all possible pages can be exhausted.

**Colab can terminate the runtime.** There is no keep-alive/limit-evasion trick or guaranteed 24/7 run. Verified Drive checkpoints allow another Run all with the same project to continue; work since the last completed remote snapshot can be lost. Use only **one runtime per Drive project**. Keep your input, Drive project and notebook outputs private.


In [ ]:
# @title 1. Ready-to-run defaults (no changes needed)
PROJECT_NAME = "publicmail-33144"  # @param {type:"string"}
# Only change PROJECT_NAME to match an existing differently named collection.
# Advanced input alternatives remain available; the default uploads once and resumes.
USE_GOOGLE_DRIVE = True
INPUT_MODE = "Upload or reuse saved queue"
DRIVE_INPUT_PATH = ""
SQLITE_INPUT_TABLE = ""
ENABLE_BROWSER = True
EXPECTED_DOMAINS = 33144
CODE_REF = "main"

from pathlib import Path
CONTENT_ROOT = Path("/content")
print("AUTOMATIC: whole list → deeper contacts → transient recovery; no pilot or planned time stop.")
print("Use the SAME project name when resuming. CPU only; Chromium is installed automatically.")


## 2. Install / automatically update the private application

Setup fetches the latest full-list workflow every time, so an older reused Colab checkout cannot silently run its old 100-claim pilot. Before merge it automatically falls back to the published feature branch. It prints the exact commit and checks workflow version 2 or later.

Application packages and Chromium go into a **private environment**, not Colab's kernel. The source folder is separate from all private input/state. A clean checkout is updated without resetting or deleting any private files; local source edits are preserved by stopping with an explanation instead of overwriting them. Installation may take several minutes.


In [ ]:
# @title Install / update the private runtime
import os
import re
import subprocess
import sys

if "client" in globals():
    client._idle()  # Do not replace packages underneath an active worker.
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,63}", PROJECT_NAME):
    raise ValueError("Project name must use letters/numbers/underscore/hyphen, up to 64 characters.")
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_./-]{0,127}", CODE_REF):
    raise ValueError("Choose a valid Git branch or tag.")
CONTENT_ROOT.mkdir(parents=True, exist_ok=True)
REPOSITORY = CONTENT_ROOT / "publicmail-code"
REPO_URL = "https://github.com/FredTheFork/emailscraperfromsites.git"
COLAB_FEATURE_REF = "arena/ff9cb618-emailscraperfromsites"
GIT_ENV = dict(os.environ, GIT_TERMINAL_PROMPT="0")
if not REPOSITORY.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", CODE_REF, REPO_URL, str(REPOSITORY)], check=True, env=GIT_ENV)
elif not (REPOSITORY / ".git").is_dir():
    raise RuntimeError("The source folder is not a Git checkout. Keep your Drive project; use a fresh runtime.")
dirty = subprocess.check_output(["git", "-C", str(REPOSITORY), "status", "--porcelain"], text=True).strip()
if dirty:
    raise RuntimeError("Local SOURCE edits detected; not overwriting them. Preserve those edits and use a fresh runtime with the SAME Drive project.")
def fetch_source(ref):
    subprocess.run(["git", "-C", str(REPOSITORY), "fetch", "--depth", "1", "origin", ref], check=True, env=GIT_ENV)
def has_full_workflow():
    try:
        source = subprocess.check_output(["git", "-C", str(REPOSITORY), "show", "FETCH_HEAD:src/publicmail/coverage.py"], text=True, stderr=subprocess.DEVNULL)
        return bool(re.search(r"WORKFLOW_VERSION\s*=\s*([2-9]|[1-9][0-9]+)\b", source))
    except subprocess.CalledProcessError:
        return False
fetch_source(CODE_REF)
if not has_full_workflow():
    if CODE_REF != "main":
        raise RuntimeError("This source revision is too old for automatic full-list collection.")
    print("main does not include full-list workflow v2 yet; automatically using its published feature branch.")
    fetch_source(COLAB_FEATURE_REF)
    if not has_full_workflow():
        raise RuntimeError("Updated full-list workflow is missing. No input/state was changed.")
subprocess.run(["git", "-C", str(REPOSITORY), "checkout", "--detach", "FETCH_HEAD"], check=True, env=GIT_ENV)
CODE_REVISION = subprocess.check_output(["git", "-C", str(REPOSITORY), "rev-parse", "HEAD"], text=True).strip()
print("Using source revision:", CODE_REVISION)
sys.path.insert(0, str(REPOSITORY / "notebooks"))
from colab_client import ColabClient, install_runtime
PYTHON, APP_ENV = install_runtime(REPOSITORY, CONTENT_ROOT, browser=ENABLE_BROWSER)
LOCAL_PROJECT = CONTENT_ROOT / "publicmail-workspaces" / PROJECT_NAME
LOCAL_PROJECT.mkdir(parents=True, exist_ok=True)
print("Private runtime ready:", PYTHON)


## 3. Mount Drive and restore saved progress

Approve the normal Google Drive authorization dialog in Colab. Your project is stored privately under **MyDrive / PublicMail / project-name**. Share neither that folder nor its results publicly.

The **active SQLite database stays on `/content`**, Colab's local disk. Drive receives verified, closed snapshots—not an active WAL database. Three good snapshot generations are kept. A session marker catches many accidental duplicate runtimes, but Drive is not a distributed lock service: **never run two Colab runtimes on this project simultaneously**.

If Drive is disabled, all files are ephemeral and cannot survive a runtime reset. This notebook does not create a fake backup folder when the required Drive mount is missing.

In [ ]:
# @title Mount Drive / restore the queue
import json

DRIVE_MOUNT = None
DRIVE_PROJECT = None
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    DRIVE_MOUNT = CONTENT_ROOT / "drive"
    drive.mount(str(DRIVE_MOUNT))
    if not os.path.ismount(DRIVE_MOUNT) or not (DRIVE_MOUNT / "MyDrive").is_dir():
        raise RuntimeError("Drive is not mounted correctly. Re-run this cell; do not proceed without your backups.")
    DRIVE_PROJECT = DRIVE_MOUNT / "MyDrive" / "PublicMail" / PROJECT_NAME
else:
    print("WARNING: Drive is disabled. A runtime reset loses the input, database, snapshots, and exports.")

client = ColabClient(PYTHON, LOCAL_PROJECT, drive_dir=DRIVE_PROJECT, drive_mount=DRIVE_MOUNT, environment=APP_ENV)
health = client.json("doctor", *(["--check-browser"] if ENABLE_BROWSER else []))
if health.get("workflow_version", 0) < 2:
    raise RuntimeError("Stale application detected. Rerun installation; this version will not start a pilot.")
print("Environment:", json.dumps(health, indent=2))
PREPARED = client.json("prepare")
print("Restored snapshot:", PREPARED["restored"] or "Existing local state / new collection")
print("Backup storage:", PREPARED["backup_dir"])
print("Domains saved:", format(PREPARED["stats"]["total"], ","))
print("Queue is paused. No target-site crawl has started from this cell.")

## 4. Import your actual CSV or SQLite list

On the first run, select your original **`domains.csv` containing 33,144 domains**. It is uploaded to the Colab runtime, imported in repeat-safe batches, and copied privately to your Drive project. A resumed queue already contains its original metadata and does not require another upload.

For a large SQLite file, place a **consistent backup/export** in Drive and select **Use a Drive file path** above. Do not upload only the main file of a database whose writer still has an uncheckpointed WAL. Input databases are copied onto local disk before querying and are never used as the working state DB.

In [ ]:
# @title Upload / reuse / import input
current = client.json("status")
input_path = None
if INPUT_MODE == "Use a Drive file path":
    if not DRIVE_INPUT_PATH.strip():
        raise ValueError("Enter the full private Drive input path in Collection settings, then rerun that settings cell.")
    input_path = Path(DRIVE_INPUT_PATH).expanduser()
elif INPUT_MODE == "Upload or reuse saved queue" and current["stats"]["total"]:
    print("Reusing", format(current["stats"]["total"], ","), "saved domains; no upload needed.")
elif INPUT_MODE == "Upload or reuse saved queue" and PREPARED.get("saved_input"):
    input_path = Path(PREPARED["saved_input"])
else:
    from google.colab import files
    uploads_dir = LOCAL_PROJECT / "uploads"
    uploads_dir.mkdir(exist_ok=True)
    uploaded = files.upload(target_dir=str(uploads_dir))
    if len(uploaded) != 1:
        raise ValueError("Choose exactly one CSV or SQLite input, then rerun this cell.")
    input_path = uploads_dir / Path(next(iter(uploaded))).name
    del uploaded  # Do not hold a second copy of a large upload in notebook memory.
if input_path is not None:
    args = ["import", str(input_path)]
    if SQLITE_INPUT_TABLE.strip():
        args += ["--table", SQLITE_INPUT_TABLE.strip()]
    report = client.json(*args)
    print("Import report:", json.dumps(report, indent=2))
STATE = client.json("status")
total = STATE["stats"]["total"]
if not total:
    raise RuntimeError("No domains were imported. Read the import error; do not start an empty run.")
print("READY:", format(total, ","), "domains;", format(STATE["stats"]["unique_emails"], ","), "emails collected so far.")
if EXPECTED_DOMAINS and total != EXPECTED_DOMAINS:
    print("CHECK INPUT: expected", format(EXPECTED_DOMAINS, ","), "domains, but the saved queue contains", format(total, ","))
print("A verified recovery snapshot has been saved. Importing does not start crawling.")

## 5. Automatic collection — the long-running cell

**Do not change any options.** It runs the entire pending website queue, not a sample, then automatically applies:

1. **Whole list:** 12 concurrent sites, at least 1.5 seconds between requests to the same site, up to 40 content URLs / depth 3, sitemaps, text PDFs and two rendered pages.
2. **Deeper contacts:** no-email sites get a refreshed public-page pass; sites with limits, errors or unrendered contact pages continue their saved frontier. Budgets automatically increase to 80 content URLs / depth 5 and four rendered pages. Existing evidence and denied paths are retained.
3. **Transient recovery:** one additional bounded round for unreachable/transient failures and exhausted rate limits, respecting persistent server cooldowns. Robots denials, authentication blocks and CAPTCHA challenges are **not** requeued to evade them.

Stages and selections are saved atomically in the database and checkpoints. **Run all again after a Colab reset automatically resumes the same stage**, rather than restarting completed passes. Running it after a finished plan makes no new website requests.

Progress logs appear about every 30 seconds. Verified snapshots are saved before requests, between stages, about every two minutes after the previous backup finishes, and at graceful exit. A failed backup stops new claims. Final exports include an unmistakable completion report, source evidence and an outcome for **every input website**.

There is **no default five-hour stop**, but Google can end its runtime; this workload can take multiple sessions and days. A long Retry-After may leave it correctly waiting. To pause deliberately, interrupt **once** and allow final backup/export. Never reset/delete the runtime to pause or start another copy concurrently.


In [ ]:
# @title Automatically collect / resume EVERY queued website
print("Starting the FULL queue and automatic follow-ups. No 100-site pilot; no planned time stop.")
client.run("run", "--browser" if ENABLE_BROWSER else "--no-browser")
print("Collection stopped safely. The completion report below says whether work is still unfinished.")


## 6. Completion report and result download

**Read `READ_ME_FIRST.txt` in the ZIP first.** `INCOMPLETE` means queued/unstarted websites or automatic follow-ups still remain. A small address count is never presented as proof the whole list was processed. Finished outcomes can still include access blocks, dead sites, no-email sites, errors and finite limits; these are reported, not hidden.

The ZIP contains `emails.csv` (website/email associations), `unique_emails.csv`, `observations.csv`, exact `observations.jsonl`, **`coverage.csv` (every input website)**, `summary.json`, `run_report.json` and `READ_ME_FIRST.txt`.

**Website count, websites with an email and unique email count are different.** `completed` only means evidence was found. Addresses are observed public data, not verified ownership, deliverability or marketing consent. Downloads do not display collected addresses in notebook output by default.

Results are copied privately to **MyDrive / PublicMail / project-name / exports**. After a hard reset, Run all restores the saved queue. You can also rerun just status/download after restoring, to export partial progress without waiting for the whole collection.


In [ ]:
# @title Current completion status — never confuse a partial run with a full result
from datetime import datetime, timezone

STATUS = client.json("status")
s = STATUS["stats"]
r = STATUS["collection"]
print("COLLECTION:", r["state"], "| Stop reason:", r["stop_reason"])
print(f"Websites: {s['total']:,} | STARTED: {s['sites_started']:,} | NEVER STARTED: {s['sites_unstarted']:,}")
print(f"Queue still pending / retry / active: {r['completion']['queue_remaining']:,}")
print(f"Websites with email: {s['with_email']:,} | Unique addresses: {s['unique_emails']:,} | Website-email pairs: {s['site_email_pairs']:,}")
print("Outcomes:", json.dumps(s["statuses"], indent=2))
print(r["next_step"])
if s.get("next_retry_at"):
    try:
        print("Next retry eligible:", datetime.fromtimestamp(s["next_retry_at"], timezone.utc).isoformat())
    except (ValueError, OverflowError, OSError):
        print("Next retry is outside supported date range; inspect Retry-After in coverage.csv.")
if STATUS.get("last_checkpoint"):
    last = STATUS["last_checkpoint"]
    print("Last verified snapshot:", datetime.fromtimestamp(last["created_at"], timezone.utc).isoformat())
print("Mode:", s["mode"], "| Worker:", s["worker_state"])
if s.get("worker_error"):
    print("WORKER NEEDS ATTENTION:", s["worker_error"])


In [ ]:
# @title Automatically export current progress / download the result ZIP
import zipfile

marker = LOCAL_PROJECT / "last-export.json"
try:
    RESULTS = client.json("export")  # Never silently prefer a stale pilot ZIP.
except RuntimeError as error:
    print("EXPORT NEEDS ATTENTION:", error)
    if not marker.is_file():
        raise
    RESULTS = json.loads(marker.read_text())
    print("Recovering the last LOCAL ZIP; it may predate current progress. Its own status follows.")
archive = Path(RESULTS["export_zip"])
if not archive.is_file():
    raise RuntimeError("No local result ZIP is available. Keep your Drive snapshots and retry export after restoring.")
with zipfile.ZipFile(archive) as zipped:
    if "READ_ME_FIRST.txt" in zipped.namelist():
        print(zipped.read("READ_ME_FIRST.txt").decode("utf-8"))
    else:
        print("OLD EXPORT WITHOUT COMPLETION REPORT — this is not evidence of a full-list run.")
print("ZIP:", archive, "|", round(archive.stat().st_size / 1024**2, 2), "MiB")
print("Private Drive copy:", RESULTS.get("drive_export_zip") or "Not copied to Drive; download while this runtime survives.")
from google.colab import files
files.download(str(archive))


## Automatic pass history — no manual requeue needed

Deeper collection and transient recovery now run automatically. They are finite and based on saved coverage, **not an arbitrary target email count**. A finished plan is not restarted merely by rerunning the notebook. Blocked paths and historical observations remain in the state database.

If anything remains unfinished, keep the same project and Run all again. It will restore and continue. For diagnostics inspect the pass history below and `run_report.json`; do not erase the queue or repeatedly requeue access-restricted sites.


In [ ]:
# @title Automatic coverage history
HISTORY = client.json("status")["collection"]
plan = HISTORY.get("automatic_plan")
if plan:
    for stage in plan["stages"]:
        print(stage["name"], "—", stage["state"], "| selected:", stage.get("selected", "not yet"), "| unique emails after pass:", stage.get("unique_emails", "in progress"))
else:
    print("Automatic collection has not started yet.")
print(HISTORY["next_step"])


## Recovery and troubleshooting

- **Colab reset / lost connection:** use the same project name, mount Drive again, run setup/restore/import. Run all; automatic full-list coverage resumes without changing options. Already saved evidence is not erased.
- **Another runtime/session marker:** stop the other runtime. If it died, wait up to 15 minutes for the advisory marker to expire. Never delete a marker just to run concurrently.
- **Drive quota / mount / I/O failure:** the runner stops rather than silently claiming continuing protection. Download the local ZIP if possible, fix Drive, and rerun. Keep previous snapshot `.db` + `.json` pairs together. Do not copy an active database/WAL to Drive manually.
- **Corrupt latest snapshot:** recovery tries older verified generations. If all are invalid, it refuses an empty reset. Keep those files; do not delete your project to clear the message.
- **Low space:** delete only older result ZIPs/exports you have safely copied elsewhere. Three complete DB generations plus staging require several times the DB's size. A Drive volume/quota is not an off-account backup.
- **Browser installation/launch error:** rerun installation and the environment check, or disable optional browser fallback and rerun those cells. Missing Chromium must not turn the entire input into failed sites.
- **No public email / access block:** inspect coverage/evidence; do not assume a bug or attempt a bypass. Some sites cannot provide an email to this scraper.
- **Privacy:** the notebook/Drive project can contain business metadata and personal addresses. Keep it private, protect your account, clear outputs before sharing, and set suitable retention/marketing-consent policies.

For detailed limitations, see the repository's `docs/COLAB.md`, `README.md`, and `SECURITY.md`. A continuously supervised VPS remains an alternative if you later need stronger always-on guarantees, but is **not required to use this notebook**.